Generate a chart for a validation song and compare it with the ground truth

Runs top to bottom: setup, load validation data, load model, generate, export.

In [1]:
from pathlib import Path
import torch

from tracers.datasets import AutoOsu4KDataset
from tracers.project import setup_project
from tracers.datasets.factory import build_datasets, build_loader_for_model
from tracers.pipelines.trainer_factory import build_trainer

PROJECT_ROOT = Path.cwd().parent
setup = setup_project(PROJECT_ROOT)

MODEL_KEY = "tracer_global_np"

experiment_config = setup["experiment_config"]
hyperparams = setup["hyperparams"]
segment_frames = setup["segment_frames"]
batch_size = setup["batch_size"]
device = setup["device"]

In [2]:
audio_dir, beatmap_dir, train_dataset, val_dataset, test_dataset, train_ids, val_ids, test_ids = build_datasets(
    PROJECT_ROOT,
    segment_frames=segment_frames,
    seed=42,
)

In [3]:
from tracers.pipelines.generate import run_validation_generation

trainer, model = build_trainer(
    model_key=MODEL_KEY,
    hyperparams=hyperparams,
    experiment_config=experiment_config,
    device=device,
    checkpoint_path=str(PROJECT_ROOT / "notebooks" / "checkpoints" / "best.pt"),
    train_loader=None,
    valid_loader=None,
)

Loaded c:\Users\joshua\vsjycode\csc173\transformers\TRACERS\notebooks\checkpoints\best.pt (epoch 39, best val loss 0.1490)


In [4]:
context_frames = 500

val_dataset = AutoOsu4KDataset(
    beatmap_dir=beatmap_dir,
    audio_dir=audio_dir,
    segment_frames=segment_frames,
    crop_mode="all",
    stride_frames=segment_frames - context_frames,
    song_ids=val_ids,
    audio_cache_size=8,
)

val_loader = build_loader_for_model(
    val_dataset,
    batch_size=2,
    shuffle=False,
    num_workers=0,
    model_key=MODEL_KEY,
)

In [5]:
# ---------- settings ----------
CHART_ID = None          # e.g. "1055661-148-4"; None = first chart in the validation set
BPM = 120                # only used to place notes in the exported JSON
OFFSET_MS = 0
LANE_MAP = (0, 2, 4, 6)

TEMPERATURE = None        # None = greedy
TOP_K = None
TOP_P = None
SEED = 0
SAVE_DEBUG_CSV = True

# ---------- collect the chart's segments ----------
INPUT_KEYS = ["specs", "beat_phase", "beat_num", "difficulty", "valid_frames"]
segments, found = [], False
full_song_specs = None

for batch in val_loader:
    for i, cid in enumerate(batch["chart_id"]):
        if CHART_ID is None:
            CHART_ID = cid

        if cid != CHART_ID:
            if found:
                break
            continue

        found = True

        # Find the full-song spectrogram associated with this sample.
        global_idx = int(batch["global_indices"][i].item())
        song_specs = batch["global_specs"][global_idx]

        if full_song_specs is None:
            full_song_specs = song_specs

        segments.append({
            "start": int(batch["start_frame"][i]),
            "sample": {k: batch[k][i:i+1] for k in INPUT_KEYS},
            "actual": batch["actions_target"][i].cpu(),
        })
    else:
        continue
    break

segments.sort(key=lambda s: s["start"])

print(f"Chart {CHART_ID}: {len(segments)} segments")
print("Full song specs:", full_song_specs.shape if full_song_specs is not None else None )

Chart 1055661-148-4: 10 segments
Full song specs: torch.Size([3, 14946, 80])


In [6]:
model = trainer.model
device = next(model.parameters()).device


full_global_context = None

if hasattr(model, "prepare_global_context"):
    model.eval()

    with torch.no_grad():
        full_global_context = model.prepare_global_context(
            global_specs=[full_song_specs],
            global_indices=torch.tensor([0]),
            start_frames=torch.tensor([0]),
            seq_len=full_song_specs.size(1) if full_song_specs is not None else None,
            device=device,
            dtype=full_song_specs.dtype if full_song_specs is not None else None,
        )

In [7]:
generated, actual, debug_segments = run_validation_generation(
    trainer,
    val_loader,
    chart_id=CHART_ID,
    context_frames=context_frames,
    seed_action=None,
    temperature=TEMPERATURE,
    top_k=TOP_K,
    top_p=TOP_P,
    seed=SEED,
    collect_debug=SAVE_DEBUG_CSV,
)

Chart: 1055661-148-4
Windows: 10
Target frames: 14946
Window size: 2000 | Context: 500 | Stride: 1500
Window      0-  2000 | Context:    0 | New: 2000 | Generated active:   20 | Actual active:   68
Window   1500-  3500 | Context:  500 | New: 1500 | Generated active:   31 | Actual active:   41
Window   3000-  5000 | Context:  500 | New: 1500 | Generated active:   22 | Actual active:   41
Window   4500-  6500 | Context:  500 | New: 1500 | Generated active:   34 | Actual active:   36
Window   6000-  8000 | Context:  500 | New: 1500 | Generated active:   28 | Actual active:   38
Window   7500-  9500 | Context:  500 | New: 1500 | Generated active:    1 | Actual active:   48
Window   9000- 11000 | Context:  500 | New: 1500 | Generated active:   19 | Actual active:   46
Window  10500- 12500 | Context:  500 | New: 1500 | Generated active:   36 | Actual active:   52
Window  12000- 14000 | Context:  500 | New: 1500 | Generated active:   24 | Actual active:   38
Window  13500- 14946 | Context:  5

In [8]:
from tracers.export.chart import export_generated_chart

valid_all = torch.ones(len(generated), dtype=torch.bool)

output_dir = Path(f"charts/{CHART_ID}")
output_dir.mkdir(parents=True, exist_ok=True)

export_generated_chart(
    generated,
    output_dir / "generated_np_global_chart.json",
    bpm=BPM,
    start_frame=1,
    offset_ms=OFFSET_MS,
    lane_map=LANE_MAP,
)

export_generated_chart(
    actual,
    output_dir / "actual_full_chart.json",
    bpm=BPM,
    start_frame=1,
    offset_ms=OFFSET_MS,
    lane_map=LANE_MAP,
)

print("Saved generated_np_global_chart.json and actual_full_chart.json")

if SAVE_DEBUG_CSV:
    from tracers.generate.converter import export_generation_debug
    export_generation_debug(debug_segments, output_path=str(output_dir / "generation_np_global_debug.csv"), top_k=5)

Saved generated_np_global_chart.json and actual_full_chart.json
Exported 14946 frame records to: charts\1055661-148-4\generation_np_global_debug.csv
